# Phase 2 (L-NER) — Run on Phase-1 **Encoder-Decoder model 01** (bidirectional encoder), full fine-tuning

Config-driven like the other NER notebooks: edit **Cell 2** only.

### Why this model is the natural fit for NER
| | Decoder 01 / 02 / 03 | **Encoder-decoder (this notebook)** |
|---|---|---|
| context for token *i* | left only (needed tricks: echo / unmasking) | **both sides, natively**: the encoder uses full self-attention (`encoder_causal=False`) |
| pretraining signal on the encoder | next-token prediction | **BERT-style masked-LM** (15% masking, 80/10/10 rule) on 1 of every `mlm_every` micro-batches, plus the seq2seq objective through cross-attention |
| what we use at NER time | whole model | **the encoder only**; decoder, cross-attention and LM head are not needed |

This is exactly the recipe of **Module 2.4**: a BERT-style encoder, a linear layer on every token's final hidden vector `T_i`, a softmax over the 25 BIO tags. No `ATTN_MODE` choice exists here: `ATTN_MODE = "bidir"` is a fixed constant, so the shared window code builds `[BOS] + window`.

### Size (and one consequence for your KD plan)
| part | parameters |
|---|---|
| embeddings (32000 × 512, tied) | 16.38M |
| 6 encoder blocks (+ final norm) | 19.27M |
| **encoder-only NER backbone (what is trained here)** | **35.66M** |
| decoder (6 blocks) + MLM head, **unused** | 25.83M |
| full Phase-1 model | 61.49M |

**KD warning:** the capstone requires the KD teacher to be *larger* than the student. The default student of the KD notebook (Decoder-03 with 7 blocks) has 36.58M parameters, which is **more** than this 35.66M NER model, so this run cannot be its teacher (the KD notebook would stop at its assertion). Decoders 01/02/03 (≈55M) can. If you really want this model as a teacher, set `STUDENT_KEEP_LAYERS` in the KD notebook to at most 6 blocks (33.69M).

### `METHOD`
| `METHOD` | What is trained | Notes |
|---|---|---|
| `"full"` | encoder + embeddings + head | **default** (run #1 of your plan) |
| `"partial"` | top `PARTIAL_K` encoder blocks + final norm + head | embeddings and lower blocks frozen |
| `"lora"` / `"lorma"` | adapters on the 7 Linear types of every encoder block + head | same adapter code as the decoder notebooks |
| `"norm_only"` | RMSNorm gains + head | run #5 of your plan can use this notebook too |

### Windows
The encoder was trained on 768-token inputs (`encoder_len`) and its RoPE table covers 1024 positions. We use `[BOS] + 700 tokens = 701`, stride 350, so every window is inside the trained length. The notebook warns if you override it beyond 768.

### Pipeline (identical to the other NER notebooks)
document → BPE tokens with character offsets → 25 BIO tags → sliding windows → **Phase-1 encoder** → `Linear(512→25)` → average overlapping windows → BIO decode (+ learned edge-punctuation trim) → character spans → strict macro-F1.

### Outputs (in `WORK_DIR/runs/<RUN_NAME>/`)
`run_log.json`, `ner_*.pt` (trainable weights only), `submission/submission.zip`.

In [ ]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

In [ ]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "encdec_full_smoke"
METHOD     = "full"           # "full" | "partial" | "lora" | "lorma" | "norm_only"
SMOKE_TEST = True             # True: ~250 steps on a subset, dev check only, NO submission written
ATTN_MODE  = "bidir"          # FIXED: the encoder is bidirectional by construction (kept as a constant for the shared cells)
PARTIAL_K  = 3                # METHOD="partial": train the top K of the 6 encoder blocks (0 = final norm + head only)

# ---- Phase-1 checkpoint (written by caps_enc_dec_architecture_01.ipynb; must have encoder_causal=False) ----------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # same repo as Phase 1; files are Phase_2_LNER/{Train,Test}/...
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_2_LNER/ (checked first)
TRAIN_FOLDS = [1]
VAL_FOLDS   = [2]                                 # [] => no validation (use for the final fit on both folds)

# ---- Training -----------------------------------------------------------------------------
EPOCHS      = 5
BATCH       = 8                                   # sequences per micro-batch
ACCUM       = 2                                   # gradient accumulation (effective batch = BATCH*ACCUM)
WARMUP_FRAC = 0.06
GRAD_CLIP   = 1.0
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "partial":   dict(lr=2e-4, head_lr=1e-3, wd=0.01),
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")   # LoRA and LoRMA (encoder blocks)
HEAD_DROPOUT = 0.1
TRIM_PUNCT   = True                               # learn edge punctuation that never starts/ends a gold entity; trim it when decoding

# ---- Windows / class imbalance --------------------------------------------------------------
WINDOW_OVERRIDE, STRIDE_OVERRIDE = None, None     # tokens per window / stride; None => defaults below
NEG_WINDOW_KEEP = 0.3                             # keep this fraction of windows that contain no entity (per epoch)
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
O_SCALE_GRID    = (0.25, 0.4, 0.6, 0.8, 1.0, 1.25, 1.6, 2.0, 3.0)   # post-hoc scaling of P(O), tuned on dev
O_SCALE_FINAL   = 1.0                             # used when VAL_FOLDS=[] (copy the tuned value from the dev run)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_VAL_DOCS = 250, 25

# ---- Knowledge distillation (run #6): let this run serve as the TEACHER -------------------------------------------------------
EXPORT_SOFT_LABELS = False        # True (full run, SMOKE_TEST=False): save this model's per-token distributions for the KD notebook
EXPORT_FOLDS = [1, 2]             # folds to export (the student can only be distilled on folds that were exported)

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE == "bidir"
WINDOW, STRIDE = 700, 350                         # sequence = [BOS] + 700 = 701 <= 768 (the encoder's trained length)
WINDOW = WINDOW_OVERRIDE or WINDOW
STRIDE = STRIDE_OVERRIDE or STRIDE
SEQ_LEN = WINDOW + 1
assert SEQ_LEN <= 1024, f"sequence length {SEQ_LEN} exceeds the RoPE table (1024 positions)"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | encoder-only NER | window={WINDOW} stride={STRIDE} seq_len={SEQ_LEN} | "
      f"train={TRAIN_FOLDS} val={VAL_FOLDS} | smoke={SMOKE_TEST}")

In [ ]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# NER text must reach the tokenizer UNCHANGED (no clean_text / normalisation), otherwise character offsets drift.
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (its Cell 3: model, checkpoint helpers)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [ ]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

## Load the Phase-1 checkpoint

In [ ]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check its ENCODER
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
n_enc = sum(p.numel() for p in lm.encoder.parameters())                              # embeddings (shared) + encoder blocks + final norm
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} pos={lm_cfg.pos_type} | total {pc['total']/1e6:.2f}M | ENCODER-ONLY (used for NER) {n_enc/1e6:.2f}M")
assert SEQ_LEN <= lm_cfg.max_seq_len, "window longer than the RoPE table of this checkpoint"
if SEQ_LEN > lm_cfg.encoder_len:
    print(f"WARNING: sequence length {SEQ_LEN} > the encoder's trained length {lm_cfg.encoder_len} (RoPE extrapolation, expect some degradation)")

# Sanity: if the weights did not load properly, masked-token prediction would sit near ln(V) ~ 10.4.
MASK_ID = getattr(tokenizer, "special_to_id", {}).get("<mask>", 6)
if lm_cfg.mlm_head:
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _x = torch.tensor([[BOS_ID] + tokenizer.encode(_sent)], device=device)
    _pos = [p for p in (4, 9, 13) if p < _x.size(1) - 1]
    _xm = _x.clone(); _xm[0, _pos] = MASK_ID
    _sel = torch.zeros_like(_xm, dtype=torch.bool); _sel[0, _pos] = True
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _h = lm.encode(_xm, torch.ones_like(_xm))
        _lg = lm.mlm_logits_at(_h, _sel)
    _l = F.cross_entropy(_lg.float(), _x[0, _pos]).item()
    print(f"masked-LM loss on a legal sentence (3 masked tokens): {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
    assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
else:
    print("this checkpoint has no MLM head; skipping the masked-LM sanity check")

## Data, alignment and metrics
**Character offsets are everything here:** a span that is off by one character scores zero. Cell 14 contains the oracle test that proves the token↔character mapping is lossless.

In [ ]:
# Cell 13 — Phase-2 data: labels, folds, gold parsing, offset-convention check
from huggingface_hub import hf_hub_download

P2 = "Phase_2_LNER"
def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_2_LNER folder and set LOCAL_DATA_DIR.") from ex

def load_json(rel):
    return json.loads(fetch(rel).read_text(encoding="utf-8"))

SPEC_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
labels = [ln.split()[0] for ln in fetch(f"{P2}/Train/labels.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
if set(labels) != set(SPEC_LABELS):
    print("WARNING: labels.txt differs from the 12 labels in the capstone document:", labels)
LABEL_SET = set(labels)
print(len(labels), "labels:", labels)

def _text_of(v):
    return v if isinstance(v, str) else v.get("text", "")

def _parse_one(e):
    """One gold entity -> (start, end, label, text|None). Accepts [s,e,label(,text)] or dict forms."""
    if isinstance(e, dict):
        low = {k.lower(): v for k, v in e.items()}
        s  = next((low[k] for k in ("start", "start_char", "start_offset", "begin", "s") if k in low), None)
        en = next((low[k] for k in ("end", "end_char", "end_offset", "stop", "e") if k in low), None)
        lab = next((low[k] for k in ("label", "type", "tag", "entity", "category") if isinstance(low.get(k), str) and low[k] in LABEL_SET), None)
        tx = next((low[k] for k in ("text", "entity_text", "span", "mention", "value") if isinstance(low.get(k), str)), None)
    else:
        ints = [x for x in e if isinstance(x, int) and not isinstance(x, bool)]
        strs = [x for x in e if isinstance(x, str)]
        s, en = (ints[0], ints[1]) if len(ints) >= 2 else (None, None)
        lab = next((x for x in strs if x in LABEL_SET), None)
        tx = next((x for x in strs if x not in LABEL_SET), None)
    if s is None or en is None or lab is None:
        raise ValueError(f"cannot parse gold entity {e!r} -- paste this line to Claude and adapt _parse_one")
    return int(s), int(en), lab, tx

def parse_gold(raw):
    if isinstance(raw, list):                                   # list of records
        raw = {str(r.get("id", r.get("doc_id"))): r.get("entities", r.get("annotations", r.get("spans", []))) for r in raw}
    out = {}
    for did, ents in raw.items():
        if isinstance(ents, dict):
            ents = ents.get("entities", ents.get("annotations", []))
        out[str(did)] = [_parse_one(e) for e in ents]
    return out

TEXTS, GOLD_RAW = {}, {}
for f in (1, 2):
    TEXTS[f] = {str(k): _text_of(v) for k, v in load_json(f"{P2}/Train/fold{f}-text.json").items()}
    GOLD_RAW[f] = parse_gold(load_json(f"{P2}/Train/fold{f}-gold.json"))
TEST_TEXTS = {str(k): _text_of(v) for k, v in load_json(f"{P2}/Test/fold3-text.json").items()}
print({f: len(TEXTS[f]) for f in TEXTS}, "| test docs:", len(TEST_TEXTS))

# --- sanity: is this really the PHASE-2 test set? (Phase-1 test ids are the strings "1".."1000"; Phase-2 ids are long numbers like "1128835") ---
_tids = list(TEST_TEXTS)
print(f"PHASE-2 TEST SET: {len(_tids)} documents | first ids: {_tids[:3]} | train sample ids: {list(TEXTS[1])[:2]}")
if set(_tids) & (set(TEXTS[1]) | set(TEXTS[2])):
    raise RuntimeError("some test document ids also occur in the training folds: wrong test file?")
if len(_tids) >= 100 and all(i.isdigit() and int(i) <= 5000 for i in _tids):
    raise RuntimeError("these test ids look like the PHASE-1 prompt ids (1..1000), not Phase-2 document ids. "
                       "Check HF_REPO / LOCAL_DATA_DIR: the file must be Phase_2_LNER/Test/fold3-text.json.")
_first = next(iter(GOLD_RAW[1].items()))
print("sample gold:", _first[0], _first[1][:3])

# --- is the gold `end` exclusive (text[s:e]) or inclusive (text[s:e+1])? ---------------------------------------
ex = inc = tot = 0
for f in (1, 2):
    for did, ents in GOLD_RAW[f].items():
        t = TEXTS[f].get(did)
        for s, e, l, tx in ents:
            if t is None or tx is None: continue
            tot += 1; ex += (t[s:e] == tx); inc += (t[s:e + 1] == tx)
print(f"end-offset convention check over {tot} entities: exclusive matches {ex}, inclusive matches {inc}")
END_INCLUSIVE = inc > ex
if tot and max(ex, inc) / tot < 0.9:
    print("WARNING: <90% of gold entities reproduce their text from the offsets. Check for text normalisation / offset base.")
print("END_INCLUSIVE =", END_INCLUSIVE, "(predictions are converted back to this convention at submission time)")

GOLD = {}                                                        # fold -> {doc_id: [(s, e_exclusive, label)]}
bad = 0
for f in (1, 2):
    GOLD[f] = {}
    for did, ents in GOLD_RAW[f].items():
        if did not in TEXTS[f]:
            bad += 1; continue
        L = len(TEXTS[f][did]); keep = []
        for s, e, l, tx in ents:
            e = e + 1 if END_INCLUSIVE else e
            if 0 <= s < e <= L: keep.append((s, e, l))
            else: bad += 1
        GOLD[f][did] = keep
    for did in TEXTS[f]:
        GOLD[f].setdefault(did, [])
print("entities dropped for bad offsets / unknown doc:", bad)
GOLD_ALL = {did: g for f in (1, 2) for did, g in GOLD[f].items()}
ALL_TEXTS = {did: t for f in (1, 2) for did, t in TEXTS[f].items()}
assert len(GOLD_ALL) == len(GOLD[1]) + len(GOLD[2]), "document ids collide between fold 1 and fold 2"
cnt = Counter(l for f in TRAIN_FOLDS for ents in GOLD[f].values() for _, _, l in ents)
print("train-fold entity counts:", dict(cnt.most_common()))

In [ ]:
# Cell 14 — Tokenise with character offsets, BIO tags, BIO decoding, and the ORACLE ceiling test
TAGS = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
NT = len(TAGS)
TAG_PREFIX = [t.split("-", 1)[0] if t != "O" else "O" for t in TAGS]
TAG_LABEL  = [t.split("-", 1)[1] if t != "O" else None for t in TAGS]

# The BPE pre-tokeniser groups punctuation into runs (e.g. ".," or ")."), so a token can overshoot an entity edge by one
# punctuation character. We learn from the TRAIN gold which edge characters NEVER start / end an entity and trim them.
TRIM_END, TRIM_START = frozenset(), frozenset()
_END_CANDIDATES, _START_CANDIDATES = ",;:!?)]}\"'\u201d\u2019-\u2013\u2014/", "([{\"'\u201c\u2018,;:.-\u2013\u2014/"

def learn_trim_chars(gold, texts):
    ends, starts = Counter(), Counter()
    for did, ents in gold.items():
        t = texts[did]
        for s, e, l in ents:
            ends[t[e - 1]] += 1; starts[t[s]] += 1
    te = frozenset(c for c in _END_CANDIDATES if ends[c] == 0)
    ts = frozenset(c for c in _START_CANDIDATES if starts[c] == 0)
    return te, ts

def tokenise(text: str):
    ids, offs = tokenizer.encode(text, return_offsets=True)        # offsets are CHARACTER offsets into `text`
    return np.asarray(ids, dtype=np.int32), np.asarray(offs, dtype=np.int64).reshape(-1, 2)

def spans_to_tags(offs, ents):
    """Gold char spans -> one tag per token. A token belongs to an entity if their char ranges overlap.
    Returns (tags, number_of_entities_dropped_because_they_overlap_an_earlier_entity)."""
    N = len(offs); tags = np.zeros(N, dtype=np.int64); dropped = 0
    if N == 0: return tags, len(ents)
    ts, te = offs[:, 0], offs[:, 1]
    for s, e, l in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = int(np.searchsorted(te, s, side="right"))             # first token with end > s
        hi = int(np.searchsorted(ts, e, side="left"))              # first token with start >= e
        if lo >= hi or (tags[lo:hi] != 0).any():
            dropped += 1; continue
        tags[lo] = TAG2ID["B-" + l]; tags[lo + 1:hi] = TAG2ID["I-" + l]
    return tags, dropped

def tags_to_spans(text, offs, tag_ids):
    """Token tags -> character spans (start, end_exclusive, label). Lenient: I- after O/other label opens a span.
    Leading/trailing whitespace is trimmed because the tokenizer attaches the space to the FOLLOWING word;
    TRIM_START / TRIM_END remove punctuation that a fused punctuation-run token pulled into the span."""
    spans, cur = [], None
    for i, t in enumerate(tag_ids):
        t = int(t)
        if t == 0:
            if cur is not None: spans.append(cur); cur = None
            continue
        l = TAG_LABEL[t]
        if TAG_PREFIX[t] == "B" or cur is None or cur[0] != l:
            if cur is not None: spans.append(cur)
            cur = [l, i, i]
        else:
            cur[2] = i
    if cur is not None: spans.append(cur)
    out = []
    for l, a, b in spans:
        s, e = int(offs[a, 0]), int(offs[b, 1])
        while True:                                                 # trim whitespace + learned edge punctuation until stable
            s0, e0 = s, e
            while s < e and (text[s].isspace() or text[s] in TRIM_START): s += 1
            while e > s and (text[e - 1].isspace() or text[e - 1] in TRIM_END): e -= 1
            if (s, e) == (s0, e0): break
        if e > s: out.append((s, e, l))
    return out

def docs_from_texts(texts, gold=None):
    docs, dropped, t0 = [], 0, time.time()
    for did, text in texts.items():
        ids, offs = tokenise(text)
        d = dict(id=did, text=text, ids=ids, offs=offs, tags=None)
        if gold is not None:
            d["tags"], dr = spans_to_tags(offs, gold[did]); dropped += dr
        docs.append(d)
    print(f"tokenised {len(docs)} docs in {time.time()-t0:.0f}s | {sum(len(d['ids']) for d in docs)/1e6:.2f}M tokens"
          + (f" | entities dropped (nested/overlapping): {dropped}" if gold is not None else ""))
    return docs

def build_docs(folds, cache_name=None):
    texts = {did: t for f in folds for did, t in TEXTS[f].items()}
    assert len(texts) == sum(len(TEXTS[f]) for f in folds), "document ids collide between folds"
    key = (cache_name, len(texts), sum(len(t) for t in texts.values()))
    cp = WORK_DIR / f"tokcache_{cache_name}.pkl"
    if cache_name and cp.exists():
        c = pickle.load(open(cp, "rb"))
        if c["key"] == key: return c["docs"]
    docs = docs_from_texts(texts, GOLD_ALL)
    if cache_name: pickle.dump(dict(key=key, docs=docs), open(cp, "wb"))
    return docs

In [ ]:
# Cell 15 — Strict span-level metrics (exact start, end and label) + error breakdown
def strict_scores(gold, pred):
    """gold/pred: {doc_id: iterable of (start, end, label)}. Macro-F1 is averaged over labels that occur in the gold."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in set(gold) | set(pred):
        g, p = set(gold.get(did, ())), set(pred.get(did, ()))
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = dict(p=P, r=R, f1=(2 * P * R / (P + R)) if P + R else 0.0, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    present = [l for l in labels if per[l]["support"] > 0]
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mP = TP / (TP + FP) if TP + FP else 0.0; mR = TP / (TP + FN) if TP + FN else 0.0
    return dict(per=per,
                macro_f1=float(np.mean([per[l]["f1"] for l in present])) if present else 0.0,
                macro_p=float(np.mean([per[l]["p"] for l in present])) if present else 0.0,
                macro_r=float(np.mean([per[l]["r"] for l in present])) if present else 0.0,
                macro_f1_all12=float(np.mean([per[l]["f1"] for l in labels])),
                micro_f1=(2 * mP * mR / (mP + mR)) if mP + mR else 0.0)

def error_breakdown(gold, pred):
    """Why does each missed gold span count as wrong?  Answers 'boundary error or label error?'."""
    c = Counter()
    for did in gold:
        g = set(gold[did]); p = set(pred.get(did, ()))
        plist = sorted(p)
        for (s, e, l) in g - p:
            same_span = [x for x in plist if x[0] == s and x[1] == e]
            over = [x for x in plist if x[0] < e and s < x[1]]
            if same_span:                      c["wrong_label_same_span"] += 1
            elif any(x[2] == l for x in over): c["boundary_error_same_label"] += 1
            elif over:                         c["boundary_and_label_error"] += 1
            else:                              c["missed_entirely"] += 1
    return dict(c)

def print_scores(sc, title=""):
    print(f"{title} strict macro-F1 {sc['macro_f1']:.4f} | P {sc['macro_p']:.4f} R {sc['macro_r']:.4f} | "
          f"micro-F1 {sc['micro_f1']:.4f} | macro-F1 over all 12 labels {sc['macro_f1_all12']:.4f}")
    print(f"  {'label':10s} {'supp':>6s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    for l in labels:
        d = sc["per"][l]
        print(f"  {l:10s} {d['support']:6d} {d['p']:6.3f} {d['r']:6.3f} {d['f1']:6.3f}")

# ---- ORACLE test: gold -> tags -> spans must reproduce the gold. This bounds what ANY model can score. ----------------
def oracle_test(docs, gold):
    global TRIM_END, TRIM_START
    if TRIM_PUNCT:
        _te, _ts = TRIM_END, TRIM_START
        TRIM_END, TRIM_START = frozenset(), frozenset()
        base = strict_scores({d["id"]: gold[d["id"]] for d in docs},
                             {d["id"]: tags_to_spans(d["text"], d["offs"], d["tags"]) for d in docs})
        print(f"ORACLE without punctuation trimming: macro-F1 = {base['macro_f1']:.4f}")
        TRIM_END, TRIM_START = _te, _ts
    pred = {d["id"]: tags_to_spans(d["text"], d["offs"], d["tags"]) for d in docs}
    g = {d["id"]: gold[d["id"]] for d in docs}
    sc = strict_scores(g, pred)
    print(f"ORACLE (tokenisation ceiling): strict macro-F1 = {sc['macro_f1']:.4f} | micro-F1 = {sc['micro_f1']:.4f}")
    bad = [(l, sc["per"][l]["f1"], sc["per"][l]["fn"]) for l in labels if sc["per"][l]["support"] and sc["per"][l]["f1"] < 0.99]
    if bad: print("  labels below 0.99 (token boundaries do not line up with the entity edges):", bad)
    shown = 0
    for d in docs:
        miss = set(g[d["id"]]) - set(pred[d["id"]])
        for (s, e, l) in sorted(miss)[:2]:
            if shown < 5:
                print(f"  lost: {l} gold={d['text'][s:e]!r} -> tokens cover {d['text'][max(0,s-3):e+3]!r}"); shown += 1
    return sc

In [ ]:
# Cell 16 — Windows and sequence layout (pure python / numpy)
def make_windows(n_tok, W, S):
    """Overlapping windows (start, end) that cover every token."""
    if n_tok == 0: return []
    if n_tok <= W: return [(0, n_tok)]
    starts = list(range(0, n_tok - W + 1, S))
    if starts[-1] + W < n_tok: starts.append(n_tok - W)
    return [(a, a + W) for a in starts]

def build_seq(win_ids, win_tags=None):
    """Returns (input_ids, offset, labels|None). Logits at positions [offset : offset+len(win_ids)] belong to the window tokens.
       causal/bidir : [BOS] w
       echo         : [BOS] w [EOS] [BOS] w      (tags are read from the SECOND copy, which has seen the whole window)"""
    n = len(win_ids)
    if ATTN_MODE == "echo":
        seq = [BOS_ID] + list(win_ids) + [EOS_ID, BOS_ID] + list(win_ids)
        off = n + 3
    else:
        seq = [BOS_ID] + list(win_ids)
        off = 1
    lab = None
    if win_tags is not None:
        lab = [-100] * off + [int(t) for t in win_tags]
    return seq, off, lab

class WindowMerger:
    """Averages the per-token probabilities of every window that covers a token."""
    def __init__(self, docs):
        self.acc = [np.zeros((len(d["ids"]), NT), dtype=np.float32) for d in docs]
        self.cnt = [np.zeros(len(d["ids"]), dtype=np.float32) for d in docs]
    def add(self, di, a, b, p):
        self.acc[di][a:b] += p; self.cnt[di][a:b] += 1
    def result(self):
        return [self.acc[i] / np.maximum(self.cnt[i], 1)[:, None] for i in range(len(self.acc))]

def pad_batch(seqs, labs=None):
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), L), dtype=np.int64)
    lab = np.full((len(seqs), L), -100, dtype=np.int64) if labs is not None else None
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
        if labs is not None: lab[i, :len(labs[i])] = labs[i]
    return ids, mask, lab

## Encoder NER model, adaptation methods (full / partial / LoRA / LoRMA / norm-only), training

In [ ]:
# Cell 17 — Encoder NER model and the five adaptation methods
import copy

class EncNERModel(nn.Module):
    """Phase-1 ENCODER (embeddings -> 6 bidirectional blocks -> final RMSNorm) + dropout + Linear(d_model -> n_tags):
    the 'one extra output layer on token representations T_i' recipe of Module 2.4. The decoder is not used."""
    def __init__(self, encoder, d_model, n_tags, dropout):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_tags)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids, attention_mask=attention_mask)          # [B, T, d]; attention_mask = key padding mask
        return self.head(self.drop(h))                                      # [B, T, n_tags]

# ---- (c) LoRA (additive) --------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  W frozen; B = 0 at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


# ---- (d) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -----------------------------------------------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA, computed as y + (alpha/r) B(A y); B = 0 at start.
       'pi'   (LoRMA_pi): row i of BA rotated cyclically by i; B[:,0]=1, A[0,0]=1, rest of A = 0 => I(BA) = I_d at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)                                              # W0 x (frozen); the multiplier acts on the OUTPUT space
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


def build_ner_model():
    """Fresh copy of the Phase-1 encoder every call (several runs in one session do not contaminate each other)."""
    enc = copy.deepcopy(lm.encoder).cpu()
    ref = copy.deepcopy(enc) if METHOD in ("lora", "lorma") else None          # untouched copy for the identity check
    model = EncNERModel(enc, lm_cfg.d_model, NT, HEAD_DROPOUT)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # encoder + embeddings + head all trainable
    elif METHOD == "partial":
        n = len(enc.blocks)
        assert 0 <= PARTIAL_K <= n
        for p in enc.parameters(): p.requires_grad = False
        for blk in enc.blocks[n - PARTIAL_K:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in enc.norm_f.parameters(): p.requires_grad = True
        print(f"partial fine-tuning: encoder blocks {n - PARTIAL_K}..{n - 1} ({PARTIAL_K} of {n}) + final norm + head trainable; "
              f"embeddings and blocks 0..{n - PARTIAL_K - 1} frozen")
    elif METHOD in ("lora", "lorma"):
        for p in enc.parameters(): p.requires_grad = False
        if METHOD == "lora":
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
            print(f"LoRA applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        # The adapted encoder must equal the Phase-1 encoder at initialisation.
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a = enc.eval()(_ids, attention_mask=_m); b = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        enc.train(); del ref
    elif METHOD == "norm_only":
        for p in enc.parameters(): p.requires_grad = False
        n = 0
        for m in enc.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model

def class_weights(train_docs):
    cnt = np.zeros(NT)
    for d in train_docs: cnt += np.bincount(d["tags"], minlength=NT)
    w = np.ones(NT)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

In [ ]:
# Cell 18 — Inference: windows -> probabilities -> averaged per token -> BIO decode -> spans
@torch.no_grad()
def predict_probs(model, docs, batch_size=16):
    """Per-doc [n_tokens, NT] probabilities, averaged over the windows that cover each token."""
    model.eval()
    jobs = [(di, a, b) for di, d in enumerate(docs) for (a, b) in make_windows(len(d["ids"]), WINDOW, STRIDE)]
    jobs.sort(key=lambda j: j[2] - j[1])
    merger = WindowMerger(docs)
    for i in range(0, len(jobs), batch_size):
        chunk = jobs[i:i + batch_size]
        seqs, offs = [], []
        for di, a, b in chunk:
            s, o, _ = build_seq(docs[di]["ids"][a:b].tolist()); seqs.append(s); offs.append(o)
        ids, mask, _ = pad_batch(seqs)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device))
        probs = logits.float().softmax(-1).cpu().numpy()
        for k, (di, a, b) in enumerate(chunk):
            merger.add(di, a, b, probs[k, offs[k]:offs[k] + (b - a)])
    return merger.result()

def decode_all(docs, probs, o_scale=1.0):
    out = {}
    for d, p in zip(docs, probs):
        if len(p) == 0: out[d["id"]] = []; continue
        q = p.copy(); q[:, 0] *= o_scale                        # o_scale < 1 -> more entities (higher recall)
        out[d["id"]] = tags_to_spans(d["text"], d["offs"], q.argmax(1))
    return out

def tune_o_scale(docs, probs, gold, grid=O_SCALE_GRID):
    rows = []
    for s in grid:
        sc = strict_scores(gold, decode_all(docs, probs, s)); rows.append((s, sc["macro_f1"]))
    best = max(rows, key=lambda r: r[1])
    print("o_scale -> macro-F1:", " | ".join(f"{s}: {f:.4f}" for s, f in rows), f"\n  best o_scale = {best[0]}")
    return best[0], rows

In [ ]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def make_epoch_items(train_docs, rng):
    items = []
    for di, d in enumerate(train_docs):
        for a, b in make_windows(len(d["ids"]), WINDOW, STRIDE):
            if (d["tags"][a:b] != 0).any() or rng.random() < NEG_WINDOW_KEEP:
                items.append((di, a, b))
    rng.shuffle(items)
    return items

def train_ner(model, train_docs, val_docs, val_gold, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, tag_counts = class_weights(train_docs)
    print("class weights (O, then B-/I- per label):", np.round(cw.cpu().numpy(), 1).tolist())
    head_p = [p for n, p in model.named_parameters() if n.startswith("head.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("head.") or not p.requires_grad: continue
        (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=head_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(make_epoch_items(train_docs, random.Random(SEED + 1)))
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} windows/epoch (after negative sub-sampling) | {steps_per_epoch} optimiser steps/epoch | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(f1=-1.0, epoch=0, state=None)
    t_train = time.time()
    for ep in range(1, epochs + 1):
        items = make_epoch_items(train_docs, random.Random(SEED + ep))
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(len(items) / BATCH)
        for bi in range(nb):
            chunk = items[bi * BATCH:(bi + 1) * BATCH]
            seqs, labs = [], []
            for di, a, b in chunk:
                d = train_docs[di]
                s, o, l = build_seq(d["ids"][a:b].tolist(), d["tags"][a:b].tolist()); seqs.append(s); labs.append(l)
            ids, mask, lab = pad_batch(seqs, labs)
            ids, mask, lab = (torch.from_numpy(x).to(device) for x in (ids, mask, lab))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(ids, mask)
            loss = F.cross_entropy(logits.float().view(-1, NT), lab.view(-1), weight=cw, ignore_index=-100)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25*ACCUM:]):.4f} | lr x{m:.3f} | {time.time()-t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            probs = predict_probs(model, val_docs)
            sc = strict_scores(val_gold, decode_all(val_docs, probs, 1.0))
            rec.update(val_macro_f1=sc["macro_f1"], val_micro_f1=sc["micro_f1"])
            if sc["macro_f1"] > best["f1"]:
                best = dict(f1=sc["macro_f1"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(f1=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [ ]:
# Cell 20 — Build data, check the tokenisation ceiling, build the model, train
seed_everything(SEED)
if TRIM_PUNCT:
    TRIM_END, TRIM_START = learn_trim_chars({did: GOLD_ALL[did] for f in TRAIN_FOLDS for did in GOLD[f]}, ALL_TEXTS)
    print("learned edge-trim characters | end:", "".join(sorted(TRIM_END)), "| start:", "".join(sorted(TRIM_START)))
train_docs = build_docs(TRAIN_FOLDS, cache_name="train_" + "_".join(map(str, TRAIN_FOLDS)))
train_gold = {d["id"]: GOLD_ALL[d["id"]] for d in train_docs}
oracle_train = oracle_test(train_docs, train_gold)           # the best ANY model could reach with this tokenisation

val_docs, val_gold = [], {}
if VAL_FOLDS:
    val_docs = build_docs(VAL_FOLDS, cache_name="val_" + "_".join(map(str, VAL_FOLDS)))
    if SMOKE_TEST: val_docs = val_docs[:SMOKE_VAL_DOCS]
    val_gold = {d["id"]: GOLD_ALL[d["id"]] for d in val_docs}          # ONLY the docs we predict on
print(f"train docs {len(train_docs)} | val docs {len(val_docs)}")

model = build_ner_model()
history, best, stats = train_ner(model, train_docs, val_docs, val_gold,
                                 epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | val macro-F1 {best['f1']} | train {stats['train_seconds']/60:.1f} min | "
      f"peak GPU {stats['peak_gpu_gb']:.1f} GB")

In [ ]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    probs = predict_probs(model, val_docs)
    pred = decode_all(val_docs, probs, 1.0)
    n_pred = sum(len(v) for v in pred.values()); n_gold = sum(len(v) for v in val_gold.values())
    sc = strict_scores(val_gold, pred)
    print(f"loss: first-20 {stats['first_losses']:.3f} -> last-20 {stats['last_losses']:.3f}  (ratio {drop:.2f})")
    print(f"dev subset: predicted {n_pred} spans vs {n_gold} gold | strict macro-F1 {sc['macro_f1']:.4f} | micro-F1 {sc['micro_f1']:.4f}")
    print("tokenisation ceiling on train (oracle) macro-F1:", round(oracle_train["macro_f1"], 4))
    ok = drop < 0.5 and n_pred > 0 and sc["micro_f1"] > 0.1
    print("\nSMOKE VERDICT:", "PASS - the pipeline learns. Now run with SMOKE_TEST=False (set RUN_NAME, keep METHOD='full')." if ok else
          "CHECK - loss barely moved or (almost) nothing was predicted. Debug the pipeline first "
          "(check the oracle number above, the masked-LM line in Cell 12, learning rate).")
    print("(heuristic: 250 steps is short; a micro-F1 of 0.3-0.6 on the subset is normal. Zero means a bug.)")
else:
    print("not a smoke run")

In [ ]:
# Cell 22 — Dev-fold analysis: tune the O-scale, per-label scores, boundary-vs-label errors (full runs only)
O_SCALE = O_SCALE_FINAL                                      # used as-is when there is no validation fold
dev_scores = None
if not SMOKE_TEST and val_docs:
    probs_val = predict_probs(model, val_docs)
    O_SCALE, _ = tune_o_scale(val_docs, probs_val, val_gold)
    pred_val = decode_all(val_docs, probs_val, O_SCALE)
    dev_scores = strict_scores(val_gold, pred_val)
    print_scores(dev_scores, f"[dev, o_scale={O_SCALE}]")
    print("error breakdown of missed gold spans:", error_breakdown(val_gold, pred_val))
    print("oracle ceiling (train folds):", round(oracle_train["macro_f1"], 4))
else:
    print("skipped (smoke run or no validation fold)")

In [ ]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(pred_by_doc, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    sub = {}
    for did in TEST_TEXTS:
        L = len(TEST_TEXTS[did]); lst = []
        for s, e, l in pred_by_doc.get(did, []):
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= L and l in LABEL_SET
            lst.append([int(s), int(e - 1) if END_INCLUSIVE else int(e), l])     # same end-convention as the gold
        sub[did] = lst
    assert set(sub) == set(TEST_TEXTS), "submission ids differ from the official test ids"
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")      # must sit at the ROOT of the zip
    n = sum(len(v) for v in sub.values())
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} docs | {n} spans | per label:",
          dict(Counter(x[2] for v in sub.values() for x in v).most_common()))
    verify_submission(out_dir / "submission.zip")
    return sub


def verify_submission(zip_path):
    """Re-open the zip exactly like the evaluator would and check everything it checks (ids, structure, spans, labels)."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, dict), "prediction.json must be a JSON object {doc_id: [[start, end, label], ...]}"
    missing, unknown = set(TEST_TEXTS) - set(sub), set(sub) - set(TEST_TEXTS)
    assert not missing and not unknown, f"ids differ from the Phase-2 test set: {len(missing)} missing, {len(unknown)} unknown"
    for did, lst in sub.items():
        assert isinstance(lst, list), f"{did}: value must be a LIST of [start, end, label] (a dict means this is not a Phase-2 file)"
        for e in lst:
            assert isinstance(e, list) and len(e) in (3, 4), f"{did}: bad entry {e!r}"
            assert isinstance(e[0], int) and isinstance(e[1], int) and isinstance(e[2], str) and e[2] in LABEL_SET, f"{did}: bad entry {e!r}"
            s_, e_ = e[0], e[1] + (1 if END_INCLUSIVE else 0)                     # back to an exclusive end
            assert 0 <= s_ < e_ <= len(TEST_TEXTS[did]), f"{did}: offsets out of range {e!r}"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} documents | first ids {list(sub)[:3]}")
    print("   -> upload exactly THIS file to the Phase-2 Codabench page (not the Phase-1 submission.zip, which has the same name).")

if not SMOKE_TEST:
    test_docs = docs_from_texts(TEST_TEXTS)
    probs_test = predict_probs(model, test_docs)
    pred_test = decode_all(test_docs, probs_test, O_SCALE)
    submission = write_submission(pred_test, RUN_DIR / "submission")
    # spot-check: the text under a few predicted spans should look like the label says
    for did in list(submission)[:2]:
        for s, e, l in submission[did][:4]:
            e_ex = e + 1 if END_INCLUSIVE else e
            print(f"  {did} {l:9s} {TEST_TEXTS[did][s:e_ex]!r}")
else:
    print("smoke run: no submission written")

In [ ]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full tables)
log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, encoder_causal=lm_cfg.encoder_causal, arch=lm_cfg.arch_name, partial_k=PARTIAL_K, pos_type=lm_cfg.pos_type, smoke=SMOKE_TEST,
           train_folds=TRAIN_FOLDS, val_folds=VAL_FOLDS, hp=HP[METHOD], epochs=EPOCHS, batch=BATCH, accum=ACCUM,
           window=WINDOW, stride=STRIDE, neg_window_keep=NEG_WINDOW_KEEP, class_weight=CLASS_WEIGHT,
           params_total=model.n_total, params_trainable=model.n_trainable, history=history, best_epoch=best["epoch"],
           stats=stats, o_scale=O_SCALE, oracle_macro_f1=oracle_train["macro_f1"],
           dev_macro_f1=None if dev_scores is None else dev_scores["macro_f1"],
           dev_per_label=None if dev_scores is None else {l: round(v["f1"], 4) for l, v in dev_scores["per"].items()})
(RUN_DIR / "run_log.json").write_text(json.dumps(log, indent=2, default=float))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "tags": TAGS, "log": log},
           RUN_DIR / f"ner_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

In [ ]:
# Cell 25 — Export this run's per-token distributions ("soft labels") so it can serve as the KD teacher (run #6)
# Same shared BPE tokenizer in every architecture => token i of a document is the same token for every model, so these arrays can
# teach ANY student without needing this notebook's model code.  Stored as log-probabilities (float16): softmax(logp / T) is
# exactly the temperature-softened distribution softmax(z / T), because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time(); _out = {}
    for _f in EXPORT_FOLDS:
        _docs = build_docs([_f], cache_name=f"export_{_f}")
        _probs = predict_probs(model, _docs)                        # raw probabilities, windows averaged (NO O-scale)
        _out[_f] = {d["id"]: np.log(np.maximum(p, 1e-8)).astype(np.float16) for d, p in zip(_docs, _probs)}
        print(f"fold {_f}: {len(_docs)} docs, {sum(len(p) for p in _probs)/1e6:.2f}M tokens")
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, window=WINDOW, stride=STRIDE,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable),
                 dev_macro_f1=None if dev_scores is None else float(dev_scores["macro_f1"]), o_scale=float(O_SCALE),
                 train_folds=list(TRAIN_FOLDS), val_folds=list(VAL_FOLDS), epochs=int(EPOCHS), best_epoch=int(best["epoch"]))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, tags=list(TAGS), folds=_out), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.0f} MB) in {time.time()-_t0:.0f}s | teacher params {_meta['params_total']/1e6:.1f}M "
          f"| dev macro-F1 {_meta['dev_macro_f1']}")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

## What to do next

1. **First run (these defaults):** `SMOKE_TEST=True, METHOD="full"`. Read the *masked-LM loss* line in Cell 12 (proves the encoder loaded), the **ORACLE** line, and the smoke verdict. A bidirectional encoder usually learns NER quickly, so the loss should fall fast.
2. **Run #1 of your plan:** `RUN_NAME="encdec_full"`, `SMOKE_TEST=False`. Trains on fold 1, validates on fold 2, writes `submission/submission.zip`; upload it to Codabench.
3. **Run #5 of your plan (norm-only) needs no new notebook:** this one supports it (`RUN_NAME="encdec_norm_only"`, `METHOD="norm_only"`). The Decoder-01 notebook supports it too; pick one of the two. Your plan said "Decoder 01 or enc-dec".
4. **Free ablations for the report** (dev only): `partial` with `PARTIAL_K` ∈ {0, 3, 6}; `lorma` / `lora` on this same encoder; `WINDOW_OVERRIDE` 350 vs 700.
5. **Final fit:** `TRAIN_FOLDS=[1,2]`, `VAL_FOLDS=[]`, `EPOCHS=` best epoch, `O_SCALE_FINAL=` the `o_scale` printed by the dev run.

**Want this run to be the KD teacher (run #6)?** Cell 25 writes `teacher_softlabels_<RUN_NAME>.pt` (per-token log-probabilities for folds 1 and 2). Either set `EXPORT_SOFT_LABELS=True` in Cell 2 before the full run, or, after a finished full run, execute `EXPORT_SOFT_LABELS = True` in a new cell and then run Cell 25 (the trained model is still in memory). Save the output as a Kaggle dataset to use it in the KD notebook. **Note:** this encoder-only model (35.66M) is smaller than the default KD student (36.58M), so it cannot be that student's teacher.